CPU vs GPU Different Batch Sizes ---Benchmarking
============================================
 This notebook compares CPU vs GPU performance
 using a Hugging Face model (ESM2).


In [ ]:
# --------------------------------------------
# 0. Download data
# --------------------------------------------
import pandas as pd

!wget https://opig.stats.ox.ac.uk/webapps/ngsdb/unpaired/Gupta_2017/csv/SRR4431787_1_Heavy_IGHG.csv.gz
file_path = '/content/SRR4431787_1_Heavy_IGHG.csv.gz'
# Replace with your file path
df = pd.read_csv(file_path, skiprows = 1)
df = df[['sequence_alignment_aa']]  # only keep sequence column
df.to_csv("sequences.csv", index=False)

In [ ]:
# --------------------------------------------
# 1. Setup
# --------------------------------------------
!pip install -q transformers datasets torch

import torch
import time
from transformers import AutoTokenizer, AutoModel


In [ ]:
# --------------------------------------------
# 2. Check Device
# --------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(torch.cuda.get_device_name(0))


In [ ]:
# --------------------------------------------
# 3. Load Model + Tokenizer (from Hugging Face)
# --------------------------------------------
model_name = "facebook/esm2_t6_8M_UR50D"  # small, fast protein model
# Alternative: "bert-base-uncased"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Move model to device
model = model.to(device)
model.eval()



In [ ]:
# --------------------------------------------
# 4. Prepare Sample Data
# --------------------------------------------
# Protein sequences (example)

df = pd.read_csv("/content/sequences.csv")
sequences = df['sequence_alignment_aa'].dropna().tolist()
#Only use 1000 sequences
sequences = sequences[:1000]
# sequences = [
#     "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAN",
#     "GAVLILLLAVAVALAAPAAMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAN",
#     "MKADKVKFGGTSVANAERMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAN",
#     "MELERKLIANLKKMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANMKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAN"
# ] * 50  # replicate to increase workload


In [ ]:
# --------------------------------------------
# 5. Tokenization Benchmark
# --------------------------------------------
start = time.time()
inputs = tokenizer(sequences, return_tensors="pt", padding=True, truncation=True)
end = time.time()

print(f"Tokenization time: {end - start:.4f} sec")

# Move inputs to device
inputs = {k: v.to(device) for k, v in inputs.items()}



In [ ]:
# # --------------------------------------------
# # 6. Inference Benchmark
# # --------------------------------------------
start = time.time()
with torch.no_grad():
    outputs = model(**inputs)
end = time.time()

print(f"Inference time ({device}): {end - start:.4f} sec")


In [ ]:
# --------------------------------------------
# 7. CPU vs GPU Comparison Function
# --------------------------------------------
def benchmark(model, inputs, device, n_runs=5):
    times = []
    model = model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    for _ in range(n_runs):
        start = time.time()
        with torch.no_grad():
            _ = model(**inputs)
        torch.cuda.synchronize() if device == "cuda" else None
        end = time.time()
        times.append(end - start)

    return sum(times) / len(times)



In [ ]:
# # --------------------------------------------
# # 8. Run Comparison
# # --------------------------------------------
print("\nRunning CPU vs GPU benchmark...")

cpu_time = benchmark(model, inputs, "cpu")
print(f"Avg CPU time: {cpu_time:.4f} sec")

if torch.cuda.is_available():
    gpu_time = benchmark(model, inputs, "cuda")
    print(f"Avg GPU time: {gpu_time:.4f} sec")
    print(f"Speedup: {cpu_time / gpu_time:.2f}x")
else:
    print("GPU not available")



In [ ]:
# --------------------------------------------
# 9. Batch Size Experiment
# --------------------------------------------
batch_sizes = [1, 8, 32, 64, 200, 1000]

print("\nBatch size scaling test:")

for b in batch_sizes:
    test_sequences = sequences[:b]
    inputs = tokenizer(test_sequences, return_tensors="pt", padding=True, truncation=True)

    cpu_t = benchmark(model, inputs, "cpu", n_runs=1)

    if torch.cuda.is_available():
        gpu_t = benchmark(model, inputs, "cuda", n_runs=1)
        print(f"Batch {b}: CPU={cpu_t:.4f}s | GPU={gpu_t:.4f}s")
    else:
        print(f"Batch {b}: CPU={cpu_t:.4f}s")


In [ ]:
# --------------------------------------------
# 10. Inference on the entire dataset (CPU)
# --------------------------------------------
import time
import torch

# Bringing all the sequences in the original data frame
sequences = df['sequence_alignment_aa'].dropna().tolist()
sequences = sequences[:2000]

batch_size = 1000

# ---------------------------
# CPU benchmarking
# ---------------------------
cpu_start = time.time()

for i in range(0, len(sequences), batch_size):
    batch = sequences[i:i + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        _ = model.to("cpu")(**inputs)

cpu_end = time.time()
total_cpu_time = cpu_end - cpu_start

In [ ]:



# ---------------------------
# 11. Inference on the entire dataset (GPU benchmarking)
# ---------------------------
gpu_total_time = 0
gpu_success = True

if torch.cuda.is_available():

    model = model.to("cuda")

    gpu_start = time.time()

    for i in range(0, len(sequences), batch_size):
        batch = sequences[i:i + batch_size]

        try:
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True
            )

            inputs = {k: v.to("cuda") for k, v in inputs.items()}

            torch.cuda.synchronize()
            t0 = time.time()

            with torch.no_grad():
                _ = model(**inputs)

            torch.cuda.synchronize()
            t1 = time.time()

            gpu_total_time += (t1 - t0)

            del inputs
            torch.cuda.empty_cache()

        except RuntimeError as e:
            print(f"❌ OOM at batch {i//batch_size + 1}")
            gpu_success = False
            break

    gpu_end = time.time()

# ---------------------------
# Results
# ---------------------------
print("\n===== FINAL RESULTS =====")

print(f"Total sequences: {len(sequences)}")
print(f"Batch size: {batch_size}")

print(f"\nCPU total time: {total_cpu_time:.2f} sec")
print(f"CPU throughput: {len(sequences)/total_cpu_time:.2f} seq/sec")

if torch.cuda.is_available() and gpu_success:
    print(f"\nGPU compute time: {gpu_total_time:.2f} sec")
    print(f"GPU throughput: {len(sequences)/gpu_total_time:.2f} seq/sec")